# Desafio 3 — Ciência de Dados 1

**Aluno(a): José Augusto de Souza Oliveira**

Conteúdo das **Aulas 01 a 09**, até a validação cruzada aninhada (*nested CV*).

Rode cada célula e **escreva sua resposta** no campo indicado. Você **não** precisa escrever
código — o objetivo é interpretar e comparar os resultados. Os arquivos `.csv` estão nesta pasta.

> **Regra deste desafio:** toda afirmação que menciona um valor deve **trazer o valor**,
> copiado da saída da sua célula. Os números mudam de caderno para caderno.

Algumas células rodam várias buscas e podem levar um ou dois minutos. Espere o `[*]` virar
número antes de passar para a próxima.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import uniform, loguniform, randint
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict,
                                     StratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

def logistica():
    """Logistica dentro de um Pipeline (Aula 05)."""
    return Pipeline([("esc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000))])

def arvore():
    return DecisionTreeClassifier(max_depth=2, random_state=0)

def floresta():
    return RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1)

print("ambiente pronto")

## Exercício 1 — A grade e a borda da faixa  *(2,0 pontos)*

Uma `GridSearchCV` varre o `C` da logística numa faixa estreita. O código imprime a tabela inteira do `cv_results_` e, depois, a mesma busca com a faixa ampliada.

**(a)** Qual `C` venceu na primeira grade, e qual foi a nota? Onde esse valor está dentro da faixa que foi varrida?

**(b)** O que um vencedor na **borda** da faixa indica? Diga o que você faria a seguir.

**(c)** Depois de ampliar a faixa, qual `C` venceu e qual foi a nota? A primeira grade estava certa ou errada? Explique.

**(d)** Quantos ajustes de modelo cada uma das duas grades custou? Mostre a conta.

In [ ]:
df = pd.read_csv("q_grade.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def busca_grade(valores):
    g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv,
                     scoring="accuracy", n_jobs=-1).fit(X, y)
    t = pd.DataFrame({"C": valores,
                      "media": g.cv_results_["mean_test_score"].round(4),
                      "desvio": g.cv_results_["std_test_score"].round(4)})
    return g, t

faixa1 = [0.0001, 0.001, 0.01, 0.1]
g1, t1 = busca_grade(faixa1)
print("GRADE 1 — faixa de 0,0001 a 0,1")
print(t1.to_string(index=False))
print("melhor C:", g1.best_params_["clf__C"], "| nota:", round(g1.best_score_, 4))

faixa2 = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
g2, t2 = busca_grade(faixa2)
print("\nGRADE 2 — faixa ampliada ate 1000")
print(t2.to_string(index=False))
print("melhor C:", g2.best_params_["clf__C"], "| nota:", round(g2.best_score_, 4))

**Resposta:**

_(escreva aqui)_

## Exercício 2 — A métrica escolhe o modelo  *(1,5 pontos)*

A base `q_metrica.csv` é desbalanceada. Dois modelos — uma logística e uma árvore rasa — são avaliados com a MESMA validação cruzada estratificada, em três métricas: acurácia, F1 e average precision (AP). A última célula mostra também o que um modelo que nunca diz 1 conseguiria.

**(a)** Qual modelo vence em **acurácia** e qual vence em **F1**? Dê os quatro valores.

**(b)** A acurácia do modelo "sempre 0" é alta? Compare-a com a do melhor modelo e explique o que isso revela sobre a acurácia numa base desbalanceada.

**(c)** O que o **F1** mede que a acurácia não mede? Use a precisão e a revocação impressas para explicar por que o F1 do vencedor em acurácia é o que é.

**(d)** Qual modelo você entregaria, e com qual métrica justificaria a escolha? Cite os números.

In [ ]:
df = pd.read_csv("q_metrica.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| positivos:", int(y.sum()), f"({y.mean():.1%})")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
modelos = {"logistica": logistica(), "arvore": arvore()}

for nome, m in modelos.items():
    p = cross_val_predict(m, X, y, cv=cv)
    print(f"\n{nome}")
    print(f"  acuracia : {accuracy_score(y, p):.4f}")
    print(f"  precisao : {precision_score(y, p, zero_division=0):.4f}")
    print(f"  revocacao: {recall_score(y, p, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y, p, zero_division=0):.4f}")
    print(f"  AP       : {cross_val_score(m, X, y, cv=cv, scoring='average_precision').mean():.4f}")

zero = np.zeros_like(y)
print(f"\nmodelo que NUNCA diz 1 -> acuracia {accuracy_score(y, zero):.4f}"
      f" | F1 {f1_score(y, zero, zero_division=0):.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 3 — Grade e sorteio com o mesmo orçamento  *(1,5 pontos)*

Dois hiperparâmetros da floresta, 16 medições para cada caminho: uma grade 4 × 4 e uma `RandomizedSearchCV` com `n_iter = 16`. Depois, a mesma busca aleatória repetida com 6 sementes diferentes.

**(a)** Qual caminho deu a melhor nota, e por quanto? Dê os dois valores.

**(b)** Quantos valores **distintos** de `max_features` cada caminho testou? Ligue esse número à diferença de nota.

**(c)** Olhe as 6 sementes: quanto o resultado do sorteio balança? Cite a média, o mínimo, o máximo e a fração que ficou acima da grade. Dá para declarar um vencedor comparando um número com outro?

**(d)** Se você acrescentasse um terceiro hiperparâmetro com 4 valores, quantos ajustes cada caminho passaria a custar? Mostre as duas contas.

In [ ]:
df = pd.read_csv("q_sorteio.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

grade = {"max_features": [0.2, 0.4, 0.6, 0.8],
         "min_samples_leaf": [1, 5, 10, 20]}
g = GridSearchCV(floresta(), grade, cv=cv, scoring="accuracy", n_jobs=-1).fit(X, y)
print("grade  : 16 medicoes | melhor", round(g.best_score_, 4), "|", g.best_params_)

dist = {"max_features": uniform(0.1, 0.8),
        "min_samples_leaf": randint(1, 21)}
r = RandomizedSearchCV(floresta(), dist, n_iter=16, cv=cv, scoring="accuracy",
                       random_state=0, n_jobs=-1).fit(X, y)
print("sorteio: 16 medicoes | melhor", round(r.best_score_, 4), "|",
      {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.best_params_.items()})

print("\nvalores distintos de max_features testados:")
print("  grade  :", len(set(grade["max_features"])))
print("  sorteio:", len(set(np.round(r.cv_results_["param_max_features"].data.astype(float), 8))))

notas = np.array([RandomizedSearchCV(floresta(), dist, n_iter=16, cv=cv,
                                     scoring="accuracy", random_state=s,
                                     n_jobs=-1).fit(X, y).best_score_
                  for s in range(6)])
print(f"\n6 sementes: media {notas.mean():.4f} | min {notas.min():.4f}"
      f" | max {notas.max():.4f}")
print(f"fracao acima da grade ({g.best_score_:.4f}): {np.mean(notas >= g.best_score_):.0%}")

**Resposta:**

_(escreva aqui)_

## Exercício 4 — Escolher e medir no mesmo conjunto  *(1,5 pontos)*

A base `q_escolha.csv` é cortada em treino e teste. O primeiro bloco varre sete valores de `C` **medindo cada um no teste** e fica com o melhor. O segundo bloco escolhe o `C` por validação cruzada **dentro do treino** e toca o teste uma única vez, no fim.

**(a)** Qual `C` e qual nota saem de cada um dos dois caminhos? Dê os valores e a diferença entre as duas notas finais.

**(b)** Por que a nota do primeiro caminho não é o desempenho do modelo? Use o tamanho do teste na sua explicação.

**(c)** Qual é a regra que o segundo caminho respeita, e quando o teste pode ser tocado?

In [ ]:
df = pd.read_csv("q_escolha.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=0)
print("treino:", len(y_tr), "| teste:", len(y_te))

valores = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

# CAMINHO 1 — escolher o C olhando o TESTE
no_teste = {}
for C in valores:
    m = logistica().set_params(clf__C=C).fit(X_tr, y_tr)
    no_teste[C] = m.score(X_te, y_te)
print("\nnotas NO TESTE, por C:")
for C, s in no_teste.items():
    print(f"  C = {C:<7} -> {s:.4f}")
melhor_C = max(no_teste, key=no_teste.get)
print(f"melhor olhando o teste: C = {melhor_C} com {no_teste[melhor_C]:.4f}")

# CAMINHO 2 — escolher por CV no treino e medir uma vez so
cv = StratifiedKFold(5, shuffle=True, random_state=0)
g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv, n_jobs=-1).fit(X_tr, y_tr)
honesta = logistica().set_params(clf__C=g.best_params_["clf__C"]).fit(X_tr, y_tr).score(X_te, y_te)
C_cv = g.best_params_["clf__C"]
print(f"\nescolhido por CV no treino: C = {C_cv} (nota de CV {g.best_score_:.4f})")
print(f"nota no teste, medida uma vez: {honesta:.4f}")
print(f"diferenca para o caminho 1: {no_teste[melhor_C] - honesta:+.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 5 — Os três números da mesma busca  *(2,0 pontos)*

Da mesma busca saem três números: o `best_score_`, a **nested CV** e a nota no **teste guardado**. O código produz os três e mostra as dobras de fora da nested, uma a uma.

**(a)** Dê os três números. Qual é o maior?

**(b)** Qual deles **não** pode ser reportado como desempenho do modelo, e por quê? (A resposta é de princípio, não de qual saiu maior.)

**(c)** Olhe as dobras de fora da nested, uma a uma, e o desvio. O que esse desvio diz sobre a diferença entre os três números?

**(d)** O que você escreveria no relatório final, em uma frase, usando os números do seu caderno?

In [ ]:
df = pd.read_csv("q_nested.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=0)
print("treino:", len(y_tr), "| teste:", len(y_te), "| positivos no teste:", int(y_te.sum()))

dentro = StratifiedKFold(5, shuffle=True, random_state=0)
fora   = StratifiedKFold(5, shuffle=True, random_state=1)
espaco = {"clf__C": loguniform(0.001, 1000)}
busca = RandomizedSearchCV(logistica(), espaco, n_iter=20, cv=dentro,
                           scoring="accuracy", random_state=0, n_jobs=-1)

nested = cross_val_score(busca, X_tr, y_tr, cv=fora, scoring="accuracy")
ajustada = busca.fit(X_tr, y_tr)
teste = ajustada.score(X_te, y_te)

print(f"\n1) best_score_   : {ajustada.best_score_:.4f}")
print(f"2) nested CV     : {nested.mean():.4f} +- {nested.std():.4f}")
print(f"   por dobra     : {np.round(nested, 4)}")
print(f"3) teste guardado: {teste:.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 6 — Pódio ou platô?  *(1,5 pontos)*

Uma busca varre o `C` da logística em 12 valores. O código imprime o pódio completo — média e desvio entre dobras de cada posição.

**(a)** Qual é a diferença entre o 1º e o 5º colocado? E qual é o desvio entre dobras do 1º? Dê os dois valores e compare-os.

**(b)** Isso é um pódio ou um platô? Justifique com os números.

**(c)** Qual configuração você entregaria, e por quê? (Dica: entre empatados, há um critério melhor que "ficou em primeiro".)

In [ ]:
df = pd.read_csv("q_plato.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

valores = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]
g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv,
                 scoring="accuracy", n_jobs=-1).fit(X, y)
tab = pd.DataFrame({"C": valores,
                    "media": g.cv_results_["mean_test_score"],
                    "desvio": g.cv_results_["std_test_score"]})
tab = tab.sort_values("media", ascending=False).reset_index(drop=True)
tab.index = tab.index + 1
print(tab.round(4).to_string())

dif = tab["media"].iloc[0] - tab["media"].iloc[4]
print(f"\ndiferenca 1o - 5o: {dif:.4f}")
print(f"desvio entre dobras do 1o: {tab['desvio'].iloc[0]:.4f}")

**Resposta:**

_(escreva aqui)_